In [ ]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix
from scipy import stats
import tensorflow as tf
from tensorflow.keras import layers, models
import warnings
warnings.filterwarnings('ignore')

# ─── CONFIG ───────────────────────────────────
DATASET_PATH = r"C:\Users\admin\Videos\Any Audio Converter\WAVE"
SYNTHETIC_PATH = "synthetic_data"
SAMPLE_RATE = 16000
DURATION = 1.0
IMG_HEIGHT, IMG_WIDTH = 64, 64
N_MELS = 64
BATCH_SIZE = 16
EPOCHS = 30
K_FOLDS = 5
AUG_PER_FILE = 15
CLASSES = ["identity", "request"]
SNR_LEVELS = [0, 5, 10, 15]

# ─── AUGMENTATION ─────────────────────────────────────────────────────────────
def augment_audio(y, sr):
    rate = np.random.uniform(0.8, 1.2)
    y = librosa.effects.time_stretch(y, rate=rate)
    steps = np.random.randint(-3, 3)
    y = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)
    noise = np.random.randn(len(y)) * 0.005
    y = y + noise
    y = y * np.random.uniform(0.7, 1.3)
    shift = np.random.randint(0, int(sr * 0.2))
    y = np.roll(y, shift)
    return y

def load_audio(path, sr=SAMPLE_RATE, duration=DURATION):
    y, _ = librosa.load(path, sr=sr, duration=duration)
    target_len = int(sr * duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    return y

def audio_to_melspec(y, sr=SAMPLE_RATE):
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    mel_resized = tf.image.resize(mel_db[..., np.newaxis], [IMG_HEIGHT, IMG_WIDTH]).numpy()
    return mel_resized

# ─── SYNTHETIC DATA ────────────────────────────────────────────────────────────
def generate_synthetic_data():
    os.makedirs(SYNTHETIC_PATH, exist_ok=True)
    for cls in CLASSES:
        src = os.path.join(DATASET_PATH, cls)
        dst = os.path.join(SYNTHETIC_PATH, cls)
        os.makedirs(dst, exist_ok=True)
        if not os.path.exists(src):
            print(f"[WARNING] Folder not found: {src}")
            continue
        files = [f for f in os.listdir(src) if f.endswith('.wav')]
        for fname in files:
            y = load_audio(os.path.join(src, fname))
            for i in range(AUG_PER_FILE):
                aug = augment_audio(y, SAMPLE_RATE)
                out = os.path.join(dst, f"aug_{i}_{fname}")
                import soundfile as sf
                sf.write(out, aug, SAMPLE_RATE)
    print("Synthetic data generated.")

# ─── LOAD DATASET ─────────────────────────────────────────────────────────────
def load_dataset():
    X, y_labels = [], []
    for label, cls in enumerate(CLASSES):
        src = os.path.join(DATASET_PATH, cls)
        if os.path.exists(src):
            for f in os.listdir(src):
                if f.endswith('.wav'):
                    audio = load_audio(os.path.join(src, f))
                    X.append(audio_to_melspec(audio))
                    y_labels.append(label)
        syn = os.path.join(SYNTHETIC_PATH, cls)
        if os.path.exists(syn):
            for f in os.listdir(syn):
                if f.endswith('.wav'):
                    audio = load_audio(os.path.join(syn, f))
                    X.append(audio_to_melspec(audio))
                    y_labels.append(label)
    X = np.array(X)
    y_labels = np.array(y_labels)
    print(f"Total files: {len(X)} | identity: {np.sum(y_labels==0)} | request: {np.sum(y_labels==1)}")
    return X, y_labels

# ─── MODELS ───────────────────────────────────
def build_cnn(input_shape):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inp, out)

# ─── FIX: GLU با Lambda ───────────────────────
def conformer_block(x, d_model=64, num_heads=4, kernel_size=31):
    # Feed Forward 1
    ff = layers.LayerNormalization()(x)
    ff = layers.Dense(d_model * 4, activation='swish')(ff)
    ff = layers.Dense(d_model)(ff)
    x = x + 0.5 * ff

    # Multi-Head Self Attention
    attn = layers.LayerNormalization()(x)
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)(attn, attn)
    x = x + attn

    conv = layers.LayerNormalization()(x)
    conv = layers.Dense(d_model * 2)(conv)
    gate  = layers.Lambda(lambda t: t[..., :d_model])(conv)
    value = layers.Lambda(lambda t: t[..., d_model:])(conv)
    gate  = layers.Activation('sigmoid')(gate)
    conv  = layers.Multiply()([gate, value])

    # Depthwise conv along time
    conv = layers.DepthwiseConv1D(kernel_size, padding='same', activation='swish')(conv)
    conv = layers.LayerNormalization()(conv)
    conv = layers.Dense(d_model)(conv)
    x = x + conv

    # Feed Forward 2
    ff2 = layers.LayerNormalization()(x)
    ff2 = layers.Dense(d_model * 4, activation='swish')(ff2)
    ff2 = layers.Dense(d_model)(ff2)
    x = x + 0.5 * ff2

    return layers.LayerNormalization()(x)

# ─── FIX: Reshape با -1 ───────────────────────
def build_conformer(input_shape, d_model=64):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(d_model, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Reshape((-1, d_model))(x)
    x = conformer_block(x, d_model=d_model)
    x = conformer_block(x, d_model=d_model)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inp, out)

# ─── TRAIN K-FOLD ─────────────────────────────────────────────────────────────
def train_kfold(X, y, build_fn, name):
    kf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
    accs = []
    all_preds, all_true = [], []
    for fold, (tr, val) in enumerate(kf.split(X, y)):
        model = build_fn(X.shape[1:])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        model.fit(X[tr], y[tr], epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
                  validation_data=(X[val], y[val]))
        _, acc = model.evaluate(X[val], y[val], verbose=0)
        preds = (model.predict(X[val], verbose=0) > 0.5).astype(int).flatten()
        all_preds.extend(preds)
        all_true.extend(y[val])
        accs.append(acc)
        print(f"  {name} Fold {fold+1}: {acc:.4f}")
    print(f"  {name} Mean: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    return accs, np.array(all_true), np.array(all_preds)

# ─── SNR EVALUATION ───────────────────────────────────────────────────────────
def add_noise_snr(y, snr_db):
    sig_power = np.mean(y ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = np.random.randn(len(y)) * np.sqrt(noise_power)
    return y + noise

def evaluate_snr(X_raw_audio, y, build_fn):
    results = {}
    for snr in SNR_LEVELS:
        X_noisy = []
        for audio in X_raw_audio:
            noisy = add_noise_snr(audio, snr)
            X_noisy.append(audio_to_melspec(noisy))
        X_noisy = np.array(X_noisy)
        model = build_fn(X_noisy.shape[1:])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        model.fit(X_noisy, y, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
        _, acc = model.evaluate(X_noisy, y, verbose=0)
        results[snr] = acc
    return results

# ─── VISUALIZATIONS ───────────────────────────────────────────────────────────
def plot_waveform(audio, sr, title, fname):
    plt.figure(figsize=(8, 3))
    t = np.linspace(0, len(audio)/sr, len(audio))
    plt.plot(t, audio/np.max(np.abs(audio)+1e-8))
    plt.title(title); plt.xlabel("Time (s)"); plt.ylabel("Amplitude")
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.close()

def plot_waveform_spectrogram(audio, sr, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    t = np.linspace(0, len(audio)/sr, len(audio))
    axes[0].plot(t, audio); axes[0].set_title("Waveform")
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=axes[1])
    axes[1].set_title("Spectrogram")
    fig.suptitle(title); plt.tight_layout()
    plt.savefig(fname, dpi=150); plt.close()

def plot_kfold_comparison(cnn_accs, conf_accs, fname):
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(1, K_FOLDS+1)
    ax.plot(x, cnn_accs, 'o-', label='CNN')
    ax.plot(x, conf_accs, 's-', label='Conformer')
    ax.axhline(np.mean(cnn_accs), linestyle='--', color='blue', alpha=0.5)
    ax.axhline(np.mean(conf_accs), linestyle='--', color='orange', alpha=0.5)
    ax.set_xlabel("Fold"); ax.set_ylabel("Accuracy")
    ax.set_title("K-Fold Accuracy Comparison"); ax.legend()
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.close()

def plot_melspec_comparison(X, y, fname):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for i, cls in enumerate(CLASSES):
        idx = np.where(y == i)[0][0]
        axes[i].imshow(X[idx, :, :, 0], aspect='auto', origin='lower', cmap='magma')
        axes[i].set_title(f"Mel-Spectrogram: {cls}")
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.close()

def plot_snr(cnn_snr, conf_snr, fname):
    snrs = list(cnn_snr.keys())
    plt.figure(figsize=(8, 5))
    plt.plot(snrs, [cnn_snr[s] for s in snrs], 'o-', label='CNN')
    plt.plot(snrs, [conf_snr[s] for s in snrs], 's-', label='Conformer')
    plt.xlabel("SNR (dB)"); plt.ylabel("Accuracy")
    plt.title("Accuracy vs SNR"); plt.legend()
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.close()

def plot_confusion(true, pred, title, fname):
    cm = confusion_matrix(true, pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=14)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title); plt.colorbar(im)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.close()

# ─── MAIN ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("Checking dataset structure...")
    for cls in CLASSES:
        p = os.path.join(DATASET_PATH, cls)
        if os.path.exists(p):
            wavs = [f for f in os.listdir(p) if f.endswith('.wav')]
            print(f"  {cls}: {len(wavs)} files")
        else:
            print(f"  [MISSING] {p}")

    print("\nGenerating synthetic data...")
    generate_synthetic_data()

    print("\nLoading dataset...")
    X, y = load_dataset()

    if len(X) == 0:
        print("ERROR: No data loaded. Check dataset path and folder structure.")
        exit()

    # Sample audio for visualizations
    first_file = [f for f in os.listdir(os.path.join(DATASET_PATH, CLASSES[0])) if f.endswith('.wav')][0]
    sample_audio = load_audio(os.path.join(DATASET_PATH, CLASSES[0], first_file))

    plot_waveform(sample_audio, SAMPLE_RATE, "Normalized Waveform", "Fig1_Normalized_Waveform.png")
    plot_waveform_spectrogram(sample_audio, SAMPLE_RATE, "Waveform & Spectrogram", "Fig2_Waveform_Spectrogram.png")
    plot_melspec_comparison(X, y, "Fig4_MelSpec_Comparison.png")

    print("\nTraining CNN...")
    cnn_accs, cnn_true, cnn_pred = train_kfold(X, y, build_cnn, "CNN")

    print("\nTraining Conformer...")
    conf_accs, conf_true, conf_pred = train_kfold(X, y, build_conformer, "Conformer")

    t_stat, p_val = stats.ttest_rel(cnn_accs, conf_accs)
    print(f"\nPaired t-test: t={t_stat:.4f}, p={p_val:.4f}")

    plot_kfold_comparison(cnn_accs, conf_accs, "Fig3_Classification_Results.png")
    plot_confusion(cnn_true, cnn_pred, "CNN Confusion Matrix", "Fig6_CNN_Confusion_Matrix.png")
    plot_confusion(conf_true, conf_pred, "Conformer Confusion Matrix", "Fig6_Conformer_Confusion_Matrix.png")

    print("\nEvaluating SNR...")
    raw_audios, raw_labels = [], []
    for label, cls in enumerate(CLASSES):
        p = os.path.join(DATASET_PATH, cls)
        if os.path.exists(p):
            for f in os.listdir(p):
                if f.endswith('.wav'):
                    raw_audios.append(load_audio(os.path.join(p, f)))
                    raw_labels.append(label)
    raw_labels = np.array(raw_labels)

    cnn_snr = evaluate_snr(raw_audios, raw_labels, build_cnn)
    conf_snr = evaluate_snr(raw_audios, raw_labels, build_conformer)

    plot_snr(cnn_snr, conf_snr, "Fig5_Accuracy_vs_SNR.png")

    print("\n=== SUMMARY ===")
    print(f"CNN:       {np.mean(cnn_accs):.4f} ± {np.std(cnn_accs):.4f}")
    print(f"Conformer: {np.mean(conf_accs):.4f} ± {np.std(conf_accs):.4f}")
    print(f"p-value:   {p_val:.4f}")
    print("\nSNR Results:")
    print(f"{'SNR':>6} | {'CNN':>8} | {'Conformer':>10}")
    for snr in SNR_LEVELS:
        print(f"{snr:>6}dB | {cnn_snr[snr]:>8.4f} | {conf_snr[snr]:>10.4f}")
    print("\nAll figures saved.")
